# Exact Minimum-Sensor Trends

This notebook summarizes how the exact minimum sensor budget, $N_{min}^{opt}$, changes as the number of candidate fields $K$ increases. It uses saved per-case results created by 01_solution_method_analysis.ipynb and does not rerun the exact or greedy algorithms.

It writes the trend table and figure used by final_figures.ipynb.


In [ ]:
# Import notebook dependencies and resolve the zero-error project directory.
from collections import defaultdict
from pathlib import Path
from statistics import fmean
import csv
import math

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
print("Project directory:", PROJECT_DIRECTORY)


## Prerequisite

Run 01_solution_method_analysis.ipynb first. It creates the saved case_metrics.csv file used below.


In [ ]:
# Define the saved comparison results and the trend table and figure destinations.
SUMMARY_DIRECTORY = (
    PROJECT_DIRECTORY / "results" / "algorithm_comparison"
    / "results_summary_benchmark_1d_benchmark_2d_full"
)
TABLES_DIRECTORY = SUMMARY_DIRECTORY / "tables"
FIGURES_DIRECTORY = SUMMARY_DIRECTORY / "figures"
CASE_METRICS_PATH = TABLES_DIRECTORY / "case_metrics.csv"
NMIN_TABLE_PATH = TABLES_DIRECTORY / "nmin_by_dimension_k.csv"
NMIN_FIGURE_PATH = FIGURES_DIRECTORY / "benchmark_nmin_by_k.png"

print("Case metrics source:", CASE_METRICS_PATH)
print("Trend table destination:", NMIN_TABLE_PATH)
print("Trend figure destination:", NMIN_FIGURE_PATH)


## Exact Minimum-Sensor Trend Table

For each spatial dimension and $K$, the table reports the mean, median, 95th percentile, and maximum exact minimum sensor budget.


In [ ]:
# Load per-case results, calculate the N_min trend statistics, and write the CSV table.
if not CASE_METRICS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {CASE_METRICS_PATH}. Run 01_solution_method_analysis.ipynb first."
    )

with CASE_METRICS_PATH.open("r", encoding="utf-8-sig", newline="") as input_file:
    case_rows = list(csv.DictReader(input_file))

def percentile(values, probability):
    ordered = sorted(float(value) for value in values)
    position = probability * (len(ordered) - 1)
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

nmin_groups = defaultdict(list)
for row in case_rows:
    nmin_groups[(int(row["spatial_dimension"]), int(row["K"]))].append(
        float(row["N_optimal"])
    )

nmin_rows = []
for (dimension, K), values in sorted(nmin_groups.items()):
    nmin_rows.append({
        "dimension": f"{dimension}D",
        "spatial_dimension": dimension,
        "K": K,
        "case_count": len(values),
        "mean_N_min_opt": fmean(values),
        "median_N_min_opt": percentile(values, 0.50),
        "p95_N_min_opt": percentile(values, 0.95),
        "maximum_N_min_opt": max(values),
    })

TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
with NMIN_TABLE_PATH.open("w", encoding="utf-8", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=list(nmin_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(nmin_rows)

def markdown_table(rows):
    header = (
        "| Dimension | K | Cases | Mean $N_{min}^{opt}$ | Median | 95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = [
        f"| {row['dimension']} | {row['K']} | {row['case_count']} | "
        f"{row['mean_N_min_opt']:.2f} | {row['median_N_min_opt']:.0f} | "
        f"{row['p95_N_min_opt']:.0f} | {row['maximum_N_min_opt']:.0f} |"
        for row in rows
    ]
    return header + "\n" + "\n".join(body)

print("Loaded case records:", len(case_rows))
print("Computed trend rows:", len(nmin_rows))
print("Saved N_min trend table:", NMIN_TABLE_PATH)
display(Markdown(markdown_table(nmin_rows)))


## Exact Minimum-Sensor Trend Figure

The figure compares the mean $N_{min}^{opt}$ trend across one- and two-dimensional benchmark instances.


In [ ]:
# Render and save the exact minimum-sensor trend figure for final_figures.ipynb.
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

figure, axis = plt.subplots(figsize=(6.4, 4.0))
styles = {
    "1D": {"color": "#0072B2", "marker": "o", "linestyle": "-"},
    "2D": {"color": "#009E73", "marker": "s", "linestyle": ":"},
}
for dimension in ("1D", "2D"):
    series = [row for row in nmin_rows if row["dimension"] == dimension]
    axis.plot(
        [row["K"] for row in series],
        [row["mean_N_min_opt"] for row in series],
        label=dimension,
        linewidth=1.8,
        **styles[dimension],
    )
axis.set_xlabel(r"Number of candidate fields, $K$")
axis.set_ylabel(r"Mean exact minimum sensor budget, $N_{min}^{opt}$")
axis.set_xticks(sorted({row["K"] for row in nmin_rows}))
axis.grid(alpha=0.25)
axis.legend(frameon=False)
figure.tight_layout()

FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)
figure.savefig(NMIN_FIGURE_PATH.with_suffix(".pdf"), bbox_inches="tight")
figure.savefig(NMIN_FIGURE_PATH, dpi=300, bbox_inches="tight")
print("Saved N_min trend figure:", NMIN_FIGURE_PATH)
display(figure)


## Next Step

Run final_figures.ipynb to assemble the publication-ready figures from the saved comparison and trend tables.
